In [13]:
# --- Setup: imports and display options ---
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
from sklearn import metrics,ensemble
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

<div class="jumbotron">
    <h1 class="display-1">Classification Advanced</h1>
    <hr class="my-4">
    <p>Instructor: Dr. Yan Li</p>
</div>

## Business Puzzle: The "Hard-to-Predict" Bookings


<center><img src="./img/classification/boosting_business_puzzle.png" width=85%></center>

In hotel management, predicting cancellations is rarely straightforward:
- **The "Easy" Cases:** Non-refundable bookings or those with very short lead times are easy for any simple model (like a Decision Stump) to get right.
- **The "Hard" Cases:** Large group bookings, high-value corporate accounts, or seasonal leisure travelers often follow complex, subtle patterns that a single simple model misses.

<div class="alert alert-info">
    <strong>The Business Question:</strong> How do we build a system that doesn't just settle for 80% accuracy on the "easy" cases, but systematically identifies and masters the remaining 20% of difficult cases?
</div>

<div class="alert alert-success">
    <strong>Participation Challenge:</strong> Imagine you are the Revenue Manager. You have 10 data analysts. Analyst #1 builds a basic model. Analyst #2 only looks at what Analyst #1 got wrong. Analyst #3 only looks at what #1 and #2 both got wrong. 
    <br><br>
    If you combine their opinions, would you trust this "committee" more than a single senior analyst who tries to know everything? Why?
</div>

### Basic Idea

<center><img src="./img/classification/AdaBoost.png" width=70%></center>

- Train base classifiers on different subsets of samples

- Resample so that the classifiers improve on the samples that are hard to classify
    - samples misclassified in the previous round receive a higher weight in the next round

- The final prediction is obtained by **weighted voting** over the base classifiers

### Why Boosting Works
<div class="alert alert-info">
Boosting turns many <strong>weak learners</strong> (models only slightly better than random guessing, e.g. decision stumps) into one <strong>strong learner</strong>. Each new model focuses on the mistakes of its predecessors, so the ensemble keeps refining the decision boundary.
</div>

### Bagging vs. Boosting
| Aspect | Bagging (e.g. Random Forest) | Boosting (e.g. AdaBoost, Gradient Boosting) |
|--------|------------------------------|---------------------------------------------|
| Training | parallel, on bootstrap samples | sequential, each model fixes the previous errors |
| Sample handling | equal weights, sampled with replacement | weights re-assigned: misclassified samples get heavier weight |
| Goal | reduce variance | reduce bias |
| Model combination | majority vote | weighted vote (AdaBoost) or weighted sum (Gradient Boosting) |

## Adaptive Boosting (AdaBoost) 

#### Assumptions:
- $\mathbf{D}$: the training set of sample points $\vec{x}_i\in\mathbb{R}^d$, $i=1,2,\cdots,n$
- train $K$ rounds in total; each round is indexed by $t=1,2,\cdots,K$
- $\alpha_t$: the weight of the $t$-th classifier $M_t$ trained in round $t$
- $w_i^t$: the weight of sample $\vec{x}_i$ in round $t$, with $\vec{w}^0=\frac{1}{n}\vec{1}$

#### Procedure

-  In round $t$, draw a training set $\mathbf{D}_t$ from the distribution determined by $\vec{w}^{t-1}$
- Train classifier $M_t$ on $\mathbf{D}_t$ and compute its weighted error rate on the whole training set $\mathbf{D}$
$$
\epsilon_t=\sum_{i=1}^nw_i^{t-1} \cdot I\bigl(M_t(\vec{x}_i)\ne y_i\bigr)
$$

- The weight of the classifier trained in round $t$ is

$$
\alpha_t=\frac{1}{2}\ln\bigl(\frac{1-\epsilon_t}{\epsilon_t}\bigr)
$$

> The more accurate the classifier (the smaller $\epsilon_t$), the larger its weight $\alpha_t$ in the final vote.

> AdaBoost requires each weak classifier to be **better than random guessing**: $\epsilon_t<0.5$. Only then is $\alpha_t>0$.

- Update the weight of each sample point $\vec{x}_i\in\mathbf{D}$ according to whether it was misclassified
$$
\tilde{w}_i^t=w_i^{t-1}\cdot\exp\{-\alpha_t\cdot M_t(\vec{x}_i)\cdot y_i\}
$$

> $M_t$ is the **weak** classifier trained in round $t$. Since $M_t(\vec{x}_i) and y_i\in\{-1,+1\}$, the product $M_t(\vec{x}_i)\cdot y_i$ is exactly $+1$ when the sample is correctly classified and $-1$ when it is misclassified.

> This $\pm1$ update is for **binary** labels only. For $K$ classes (SAMME): $\tilde{w}_i^t=w_i^{t-1}\cdot\exp\{\alpha_t\cdot I(M_t(\vec{x}_i)\ne y_i)\}$ and $\alpha_t=\tfrac{1}{2}\ln\tfrac{1-\epsilon_t}{\epsilon_t}+\ln(K-1)$.

- If correctly classified, take $y_i=1$ and $M_t(\vec{x}_i)=1$, then
$$
\tilde{w}_i^t=w_i^{t-1}\cdot\underbrace{\exp\{-\alpha_t\}}_{<\,1}
$$

- If misclassified, take $y_i=1$ and $M_t(\vec{x}_i)=-1$, then
$$
\tilde{w}_i^t=w_i^{t-1}\cdot\underbrace{\exp\{\alpha_t\}}_{>\,1}
$$

- Normalize the weight vector

$$
w_i^t=\frac{\tilde{w}_i^t}{\sum_j \tilde{w}_j^t}
$$

- Given the sequentially trained classifiers $M_t$, $t=1,2,\cdots,K$, and their weights $\alpha_t$, the class of a test sample $\vec{x}$ is decided by weighted voting

$$
v_j(\vec{x})=\sum_{t=1}^K\alpha_t\cdot I\bigl(M_t(\vec{x})=c_j\bigr)
$$

where $v_j(\vec{x})$ is the weighted vote for class $c_j$

The combined classifier $\mathbf{M}^K$ predicts the class of $\vec{x}$ as

$$
\mathbf{M}^K(\vec{x})=\underset{c_j}{\arg\max}\{v_j(\vec{x})\lvert j=1,2,\cdots,k\}
$$

#### Example

| Sample indices |   x   |  y  | Weights | $\hat{y}$ (x <= 3.0)? | Correct? | Updated weights |
|:--------------:|:-----:|:---:|:-------:|:-------------:|:--------:|:---------------:|
|       1        |  1.0  |  1  |  0.1    |      1        |   Yes    |     0.072       |
|       2        |  2.0  |  1  |  0.1    |      1        |   Yes    |     0.072       |
|       3        |  3.0  |  1  |  0.1    |      1        |   Yes    |     0.072       |
|       4        |  4.0  | -1  |  0.1    |     -1        |   Yes    |     0.072       |
|       5        |  5.0  | -1  |  0.1    |     -1        |   Yes    |     0.072       |
|       6        |  6.0  | -1  |  0.1    |     -1        |   Yes    |     0.072       |
|       7        |  7.0  |  1  |  0.1    |     -1        |    No    |     0.167       |
|       8        |  8.0  |  1  |  0.1    |     -1        |    No    |     0.167       |
|       9        |  9.0  |  1  |  0.1    |     -1        |    No    |     0.167       |
|      10        | 10.0  | -1  |  0.1    |     -1        |   Yes    |     0.072       |

- The stump predicts $\hat{y}=1$ when $x\le 3.0$, otherwise $\hat{y}=-1$
- Samples 7, 8 and 9 ($x=7,8,9$) have true label $y=1$ but the stump predicts $-1$, so they are **misclassified**
- The misclassified samples receive a larger weight ($0.1 \to 0.167$) while the correctly classified samples are down-weighted ($0.1 \to 0.072$)
- In the next round, the stump is forced to pay more attention to samples 7, 8 and 9

In [8]:
eps=0.3
alpha = 0.5*np.log((1-eps)/eps)
alpha
w = np.array([1/10]*10)
w
w_t = w*np.exp(-alpha*np.array([1,1,1,1,1,1,-1,-1,-1,1]))
w_t
w_t.sum()
w_t = w_t/w_t.sum()
w_t

np.float64(0.42364893019360184)

array([0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1])

array([0.06546537, 0.06546537, 0.06546537, 0.06546537, 0.06546537,
       0.06546537, 0.15275252, 0.15275252, 0.15275252, 0.06546537])

np.float64(0.9165151389911682)

array([0.07142857, 0.07142857, 0.07142857, 0.07142857, 0.07142857,
       0.07142857, 0.16666667, 0.16666667, 0.16666667, 0.07142857])

### Scikit-learn Implementation

```python
sklearn.ensemble.AdaBoostClassifier(estimator=None, n_estimators=50, learning_rate=1.0, random_state=None)
```

- `estimator`: the base classifier; defaults to a Decision Tree Stump (a tree with `max_depth=1`)
- `n_estimators`: the number of boosting rounds
- `learning_rate`: shrinkage applied *on top of* $\alpha_t$ — the effective weight of each classifier is `learning_rate·α_t`, used both in the final vote and in the sample-weight update. Lower values make boosting more conservative (more regularized), usually requiring more `n_estimators`.

> Why on top of $\alpha_t$? $\alpha_t$ is optimal only for the *current* training distribution; applying it at full strength makes the ensemble memorize noisy training points too aggressively (overfitting). Shrinking slows the fit — a variance/bias trade-off — so later rounds can keep refining the boundary in smaller steps. Smaller step ⇒ more `n_estimators` needed.

## Gradient boosting

- Currently the most popular machine learning algorithm for tabular (structured) data

- A family of variants and improvements
    - XGBoost, LightGBM, CatBoost

### Why go beyond AdaBoost?
<center><img src="./img/classification/adaboost_limitations.png" width=80%></center>

As powerful as AdaBoost is, it faces three major hurdles in real-world business applications:

1.  **The "Outlier" Sensitivity:** Because it uses an *exponential* loss — a smooth surrogate for the 0-1 loss — it tries too hard to fix mislabeled or extreme data points, which can "poison" the whole model.
2.  **Binary Labels Only:** The classic AdaBoost recipe only works for "Yes/No" classification. It cannot directly predict continuous values like hotel room prices.
3.  **Rigid Strategy:** You cannot easily change the error metric to fit your specific business goals (e.g., caring more about false positives than false negatives).

### The Gradient Boosting Solution
<center><img src="./img/classification/gradient_boosting_logic.png" width=80%></center>

Gradient Boosting (GBM) solves these by changing the game from **re-weighting samples** to **following the gradient**:

- **Any Loss Function:** Whether you are predicting a category (Classification) or a number (Regression), GBM uses the same recipe.
- **Residual Fitting:** Each new tree doesn"t just look at "wrong" samples; it tries to predict the **remaining error (residual)** of the previous trees.
- **Robustness:** By choosing a "slower" loss function (like Logistic or Huber), GBM becomes much more tolerant of noisy outliers than AdaBoost.

<div class="alert alert-info">
    <strong>The Connection:</strong> AdaBoost is actually just a special case of Gradient Boosting! If you run Gradient Boosting with an <em>exponential loss</em> function, you recover the exact same weights as AdaBoost. GBM is the broader framework that powers modern leaders like XGBoost and LightGBM.
</div>

### How Gradient Boosting Works: The "Residual" Recipe

Imagine you are playing golf. You want to get the ball into the hole (the true value):

1.  **Initial Guess:** You take a big swing. You aren't at the hole yet, but you've made progress.
2.  **Calculate the Gap (Residual):** You look at how far you are from the hole. This "gap" is your **Residual**.
3.  **Correct the Mistake:** Your next shot doesn't try to play the whole course again; it only tries to cover the **gap** left by the first swing.
4.  **Update Position:** Your new position is  $+ h_1$.
5.  **Repeat:** You keep taking small, focused shots at the remaining gap until you are close enough.

**The Iterative Learning Loop:**

<center><img src="./img/classification/gbm_procedure_flow.png" width=90%></center>

Note how the **Residual** ($y - \hat{y}$) from the previous round becomes the **Label (Target)** for the next round. Each tree is a "specialist" that only tries to correct the remaining error.

- We have already met losses: the **0-1 loss** (classification error) and **entropy / Gini** (tree impurity).
- Gradient Boosting takes one more step: you **choose the loss** $L(y,F)$ suited to the task, and each new learner moves down that loss's **gradient** — which is exactly why the gradient now appears.

#### What is a **loss function**?

Given the true label $y$ and the model's prediction $F(\vec{x})$, the loss function measures how much a wrong prediction costs:

$$ L\big(y,\,F(\vec{x})\big) \;\ge\; 0, \qquad \text{smaller = better prediction} $$

- It is the **objective we minimize** while training the model — the model is chosen to make the total loss as small as possible.
- The losses we met earlier are all instances:
    - **0-1 loss** (classification error): $L(y,\hat{y})=1$ if $\hat{y}\ne y$, else $0$
    - **entropy / Gini** (tree impurity)
    - **squared error** (regression): $L(y,F)=(y-F)^2$
- What changes between algorithms is *which* loss they minimize — Gradient Boosting lets you **choose** the loss and minimize it by moving down its gradient.

### Why "Gradient"?

In calculus, the **Gradient** tells us the direction of the steepest increase. To minimize error (Loss), we move in the **Negative Gradient** direction.

$$\text{Negative Gradient} \approx \text{Direction to fix the error}$$

- For simple Squared Error, the negative gradient is exactly the **Residual** ($y - \hat{y}$).
- For more complex goals (like predicting probabilities), the gradient is a "pseudo-residual" that tells the next tree exactly how to nudge the predictions to lower the total error.

- The loss function we want to minimize is

$$
L(y, F_M(X))=\frac{1}{N}\sum_{i=1}^N(y_i-F_M(x_i))^2
$$

where $N$ is the total number of samples

> For squared error, the negative gradient of the loss with respect to the prediction $F(x_i)$ is exactly the residual $y_i-F(x_i)$.

- For every $m\in\{1,2,\cdots,M\}$, compute the gradient of the loss with respect to the predicted value

$$
r_{im}=-\biggl[\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)}\biggr]_{F(x)=F_{m-1}(x)}, i=1,2,\cdots,N
$$

- Train a weak model $h_m(X)$ on the data whose features are $X$ and whose labels are the (negative) gradients, then update the model

$$
F_{m+1}(X)=F_m(X)+\gamma_mh_m(X)
$$

where $\gamma_m$ is the learning rate, controlling the contribution of each weak model.

### scikit-learn Implementation

```python
sklearn.ensemble.GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=None)
```

#### A Small Worked Example of Gradient Boosting

Suppose the true values are $y=[2,4,6,8]$. Start with the simplest model $F_1$ = the mean of all $y$, i.e. $F_1=5$ for every sample:
- Residuals: $y - F_1 = [-3, -1, 1, 3]$
- Train a small regression tree $h_1$ to fit these residuals
- With learning rate $\gamma_1=0.5$, the updated model is $F_2 = F_1 + 0.5 \cdot h_1$
- Repeat on the new residuals $y - F_2$; after $M$ rounds the residuals shrink and the model fits better

## XGBoost: Extreme Gradient Boosting

### Why "Extreme"? The Evolution from GBM to XGBoost

<center><img src="./img/classification/xgboost_vs_gbm.png" width=85%></center>

XGBoost isn't a whole new algorithm; it is a **highly optimized engineering masterpiece** built on top of the Gradient Boosting framework. Think of it as a "sports car" version of the standard GBM "sedan".


#### The Three Pillars of XGBoost

1.  **Smart Optimization (The "Brain"):** While standard GBM uses only the first derivative (gradient), XGBoost uses a **Second-order Taylor Expansion** (including the Hessian). This allows it to understand the "curvature" of the loss function and find the minimum much faster.
2.  **Built-in Safety (The "Brakes"):** XGBoost adds **Regularization** ($\Omega$) directly into the objective function. It penalizes overly complex trees (those with too many leaves or huge weights), which drastically reduces overfitting.
3.  **Hardware Mastery (The "Engine"):** It uses a "block structure" to find split points in parallel across CPU cores and handles missing values automatically, making it up to **10x faster** than standard implementations.

<div class="alert alert-info">
    <strong>Simplified Intuition:</strong> 
    <br>
    - <strong>GBM:</strong> Takes steps in the right direction (Gradient).
    - <strong>XGBoost:</strong> Takes smarter, more precise steps (2nd Order) and makes sure it doesn't "trip" over noisy data (Regularization).
</div>

#### The Additive Training Strategy

Just like standard Gradient Boosting, XGBoost improves the model by **adding trees one at a time**. Each new tree fits the mistakes of the previous ensemble.


### Standard GBM vs. XGBoost: The Optimization Step

<center><img src="./img/classification/xgboost_taylor.png" width=80%></center>

Why does the **Second-order Taylor Expansion** matter? 

- **Standard GBM** is like a person walking down a hill in the dark. They only know the slope (Gradient) right at their feet. They take a step in the steepest direction.
- **XGBoost** is like someone with a flashlight. By using the second derivative (Hessian), it sees the **curvature** of the hill. It knows if the slope is getting steeper or flatter, so it can take a much more precise step towards the bottom.

#### Summary: The Relation

| Feature | Standard GBM | XGBoost (The "Extreme" Upgrade) |
| :--- | :--- | :--- |
| **Optimization** | 1st Order (Gradient only) | 2nd Order (Gradient + Hessian) |
| **Overfitting** | Relies on manual tuning | Built-in Regularization ($\\Omega$) |
| **Missing Data** | Fails or needs imputation | Learns "default paths" automatically |
| **Speed** | Serial tree building | Parallel split finding |


### The Model as an Ensemble of Trees

- In a tree ensemble, the model is the sum of multiple CART trees, and the prediction is the total of the outputs of all trees.

$$\hat{y}_i = \sum_{k=1}^K f_k(x_i), \quad f_k \in \mathcal{F}_{\text{CART}}$$

where $K$ is the number of trees and $f_k$ is the $k$-th tree.

### The Objective Function: Training Loss + Regularization

> Recap: we have met **0-1 loss** (classification error), **entropy / Gini** (tree impurity), and the **squared-error loss** (gradient boosting). XGBoost formalizes the pattern: **objective = training loss + regularization**, and you may plug in any loss.

The goal of training is to choose the parameters (including the tree structure and the leaf values) that minimize the objective function:

$$\text{obj}(\theta) = \sum_{i=1}^n \ell\big(y_i, \hat{y}_i\big) + \sum_{k=1}^K\omega(f_k)$$

where $\omega(f_k)$ is the complexity of the $k$-th tree (which prevents overfitting), and the training loss (e.g. MSE for regression, logistic loss for classification) measures how well the model fits.

Common examples:
- **Mean Squared Error (MSE)**, which yields gradient boosting:

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2$$

- **Logistic loss** (used for logistic regression), which yields LogitBoost:

$$\ell(y, \hat{y}) = \sum_{i}\big[y_i \ln(1+\exp(-\hat{y}_i)) + (1-y_i) \ln(1+\exp(\hat{y}_i))\big]$$

### Tree Ensembles (CART) and Boosted Trees

- A tree ensemble model is a set of CARTs; a single tree usually cannot reach the desired performance, so the ensemble is needed to improve it.

- Random forest and boosted trees share the same "tree ensemble model" form at the prediction level, but they are trained differently:
    - Random forest trains many trees independently and averages them;
    - Boosted trees add trees one at a time, additively, to fit the residuals or the gradients.

### The Additive Training: Building the Model Piece by Piece

<center><img src="./img/classification/additive_training_metaphor.png" width=80%></center>

In traditional ML, we try to optimize the whole model at once. In XGBoost, we follow an **Additive Strategy**: we don"t change what we have already built; we just keep adding new pieces to fix the errors.

<div class="alert alert-info">
    <strong>The "Sculptor" Logic:</strong>
    <br>
    1. **Round 1:** Build the base structure (Tree 1).
    2. **Round 2:** Add a new piece (Tree 2) that fits the gaps of Round 1.
    3. **Round $t$:** Add Tree $f_t$ to the current ensemble $\hat{y}^{(t-1)}$ to get a better result $\hat{y}^{(t)}$.
</div>

Mathematically, our prediction at step $t$ is simply the old prediction plus the new tree:

$$\hat{y}_i^{(t)} = \hat{y}_i^{(t-1)} + f_t(x_i)$$

To decide what $f_t$ should look like, we minimize an **Objective Function** that balances "Fit" and "Simplicity":

$$\text{Objective}^{(t)} = \underbrace{\sum_{i=1}^n \ell(y_i, \hat{y}_i^{(t-1)} + f_t(x_i))}_{\text{Training Loss (Fit)}} + \underbrace{\Omega(f_t)}_{\text{Regularization (Simplicity)}}$$

> **Wait, where did the messy calculus go?** XGBoost uses a math trick (Taylor Expansion) to turn this complex objective into a simple quadratic equation (like $ax^2 + bx + c$). This makes it extremely fast to solve for the best possible tree $f_t$!

### Implementation

```python
!pip install xgboost

import xgboost as xgb

xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
```
- `n_estimators`: the number of boosting rounds (trees) added sequentially
- `learning_rate` ($\eta$): the shrinkage step size — same idea as in Gradient Boosting; smaller is more conservative but needs more `n_estimators`
- `max_depth`: maximum depth of each tree; deeper trees are more complex (overfitting risk)
- `subsample`: fraction of training rows sampled in each round (row subsampling, reduces overfitting)
- `colsample_bytree`: fraction of features sampled for each tree (feature subsampling)
- `reg_lambda` (L2) and `reg_alpha` (L1): penalties on the leaf weights — these are the $\Omega(f_t)$ regularization from the objective
- `random_state`: random seed for reproducible results

Practical hints on setting them:
- start from the **defaults**: `reg_lambda=1`, `reg_alpha=0`, `subsample=1`, `colsample_bytree=1` — they work well out of the box
- **overfitting** (test much worse than training)?  *increase* `reg_lambda` (e.g. $1\to 5$) and lower `subsample` / `colsample_bytree` to $\approx 0.8$ — all of them trade a little bias for lower variance
- **underfitting** (both errors high)?  do the opposite: lower regularization and raise `subsample` / `colsample_bytree` back toward $1$
- with **very many features**, `colsample_bytree` $\approx 0.5$ adds feature diversity (like Random Forest); with **few features**, keep it close to $1$
- `reg_alpha` (L1) is mostly useful for **sparse** leaf weights or many redundant features; otherwise leave it at $0$
- tune all of these *together* with `learning_rate`, `n_estimators` and `max_depth` using cross-validation (e.g. `GridSearchCV`), because they interact

In [ ]:
# --- XGBoost (comment/uncomment the pip install as needed) ---
# !pip install xgboost
import xgboost as xgb
xgb_model = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=10)
xgb_model.fit(X_train, y_train)
print(f'XGBoost accuracy: {metrics.accuracy_score(y_test, xgb_model.predict(X_test)):.3f}')

### Key Takeaways
| Aspect | Summary |
|--------|---------|
| **What** | Boosting trains weak learners sequentially, focusing on previously misclassified samples |
| **AdaBoost** | re-weights samples; base learner defaults to a decision stump; combines by weighted vote |
| **Gradient Boosting** | fits the negative gradient (residuals) with small trees; combines by weighted sum |
| **When to use** | tabular data, when a single decision tree underfits, or as a strong baseline for competitions |
| **Popular libraries** | `AdaBoostClassifier`, `GradientBoostingClassifier`, `XGBoost`, `LightGBM`, `CatBoost` |